# Household Deprivation Data Understanding

In [ ]:
import os
import pandas as pd
import re

In [ ]:

data_path = "/Users/trivenidhamdhere/Documents/MSc Data Science/Visual Analytics/Final Coursework/Final Data/"

## 1. Inspect first 12 rows of 2011 file (no header)

In [ ]:
# 1. Look at the first 12 rows with no header parsing
"""
    Read the first 12 rows of the 'Data' sheet from the 'Households by deprivation dimensions_2011.xlsx' file.

    The Excel file is located in the `data_path` directory, and the sheet is read without interpreting
    any row as column headers (`header=None`). Only the first 12 rows are loaded to reduce memory usage
    and allow inspection of raw data structure.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the raw, unparsed first 12 rows of the specified sheet.
        Columns are integer indices (0, 1, 2, ...) since no header was provided.
"""
raw = pd.read_excel(data_path+ "Households by deprivation dimensions_2011.xlsx",
                    sheet_name="Data", header=None, nrows=12)
raw

## 2. Inspect raw 2021 data structure

In [ ]:
# 2. Now try the 2021 version — notice the header row is different!
"""
    Read the first 12 rows of the 'Data' sheet from the 'Households by deprivation dimensions_2021.xlsx' file.

    Unlike the 2011 version, the 2021 file has a different header row structure. 
    To avoid any misinterpretation, the file is read without using any row as column headers (`header=None`). 
    Only the first 12 rows are loaded to enable raw inspection and comparison with the 2011 layout.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the raw, unparsed first 12 rows of the 2021 sheet.
        Columns are integer indices (0, 1, 2, ...) because no header was specified.
"""

raw21 = pd.read_excel(data_path + "Households by deprivation dimensions_2021.xlsx",
                      sheet_name="Data", header=None, nrows=12)
raw21

# 3. Define helper to load and clean a file

In [ ]:
# 3. 
 
def load_file(filename, header_row):
    """
        Load and clean a deprivation dimensions Excel file for a given year.

        The function reads a specific sheet ("Data") from an Excel file located in the
        global `data_path` directory. After reading, it standardizes the first column
        name to "LA", removes a metadata row that starts with "In order to", strips
        whitespace from LA names, and drops any rows where LA is missing (NaN).

        Parameters
        ----------
        filename : str
            Name of the Excel file (e.g., "Households by deprivation dimensions_2011.xlsx").
        header_row : int or None
            Row number to use as column headers (0-indexed). Passed directly to
            `pd.read_excel(..., header=header_row)`. Use `None` if the file has no header.

        Returns
        -------
        pd.DataFrame
            Cleaned DataFrame with:
            - First column renamed to "LA"
            - No rows where the "LA" column starts with "In order to"
            - No rows with missing "LA" values
            - Reset index (sequential starting from 0)
    """
    df = pd.read_excel(data_path + filename, sheet_name="Data", header=header_row)
    
    df = df.rename(columns={df.columns[0]: "LA"})
    
    mask = df["LA"].astype(str).str.startswith("In order to")
    df = df[~mask].copy()
    
    df["LA"] = df["LA"].astype(str).str.strip()
    df = df.dropna(subset=["LA"])
    
    return df.reset_index(drop=True)

print("Function defined.")

# 4. Lowercase LA names (step 1 of cleaning)

In [ ]:
dep11 = load_file("Households by deprivation dimensions_2011.xlsx", header_row=8)
dep21 = load_file("Households by deprivation dimensions_2021.xlsx", header_row=7)

hlt11 = load_file("General health_2011.xlsx", header_row=8)
hlt21 = load_file("General health_2021.xlsx", header_row=7)

eco11 = load_file("Economic activity_2011.xlsx", header_row=8)
eco21 = load_file("Economic activity_2021.xlsx", header_row=7)

qua11 = load_file("Highest Qualification Total 2011_Nomis.xlsx",header_row=8)
qua21 = load_file("Highest level of qualification_2021.xlsx",header_row=7)

print("Shapes:")
print(f"  dep11: {dep11.shape}, dep21: {dep21.shape}")
print(f"  hlt11: {hlt11.shape}, hlt21: {hlt21.shape}")
print(f"  eco11: {eco11.shape}, eco21: {eco21.shape}")
print(f"  qua11: {qua11.shape}, qua21: {qua21.shape}")

In [ ]:
print("dep11 columns:", dep11.columns.tolist())
dep11.head(3)

In [ ]:
print("hlt11 columns:", hlt11.columns.tolist())
hlt11.head(3)

In [ ]:
print("eco11 columns:", eco11.columns.tolist())
eco11.head(3)

In [ ]:
print("qua11 columns:", qua11.columns.tolist())
eco11.head(3)

# 5. Remove alphanumeric prefixes (step 2)

In [ ]:
# STEP 1 — Convert all LA names to lowercase

def lowercase_la(df):
    """
        Convert all entries in the 'LA' column to lowercase and strip surrounding whitespace.

        This function ensures consistent casing for local authority names, making it easier
        to compare or merge datasets from different years (e.g., 2011 vs 2021) that may use
        inconsistent capitalization.

        Parameters
        ----------
        df : pd.DataFrame
            A DataFrame that must contain a column named 'LA'. The column values will be
            converted to strings, stripped of leading/trailing whitespace, and lowercased.

        Returns
        -------
        pd.DataFrame
            The same DataFrame with the 'LA' column transformed as described. The original
            DataFrame is modified in place and also returned for convenience.

        Examples
        --------
        >>> df = pd.DataFrame({'LA': ['  Leeds  ', 'MANCHESTER']})
        >>> lowercase_la(df)
        >>> df['LA'].tolist()
        ['leeds', 'manchester']
    """
    df["LA"] = df["LA"].astype(str).str.strip().str.lower()
    return df

dep11 = lowercase_la(dep11)
dep21 = lowercase_la(dep21)
hlt11 = lowercase_la(hlt11)
hlt21 = lowercase_la(hlt21)
eco11 = lowercase_la(eco11)
eco21 = lowercase_la(eco21)
qua11 = lowercase_la(qua11)
qua21 = lowercase_la(qua21)


# STEP 2 — Remove prefixes using regex (uacounty09:, gor:, etc.)

def remove_prefix(df):
    """
        Remove alphanumeric prefix followed by a colon from the start of each 'LA' entry.

        The function uses a regular expression to strip patterns like "uacounty09:",
        "gor:", "e07000022:", etc., which are commonly present in raw Census or ONS data.
        Any remaining whitespace around the cleaned name is also trimmed.

        Parameters
        ----------
        df : pd.DataFrame
            A DataFrame that must contain a column named 'LA'. Each value in that column
            is converted to a string, then any prefix matching the regex r"^[a-z0-9]+:"
            at the beginning is removed.

        Returns
        -------
        pd.DataFrame
            The same DataFrame with cleaned 'LA' column (modified in place and returned).

        Examples
        --------
        >>> df = pd.DataFrame({'LA': ['uacounty09:Leeds', 'gor:Manchester']})
        >>> remove_prefix(df)
        >>> df['LA'].tolist()
        ['leeds', 'manchester']

        Notes
        -----
        This function should typically be applied *after* `lowercase_la` to ensure the
        prefix regex matches lowercase characters. It does not remove colon-separated
        prefixes that appear in the middle or at the end of a string.
    """
    
    df["LA"] = df["LA"].astype(str).str.replace(r"^[a-z0-9]+:", "", regex=True).str.strip()
    return df

dep11 = remove_prefix(dep11)
dep21 = remove_prefix(dep21)
hlt11 = remove_prefix(hlt11)
hlt21 = remove_prefix(hlt21)
eco11 = remove_prefix(eco11)
eco21 = remove_prefix(eco21)
qua11 = remove_prefix(qua11)
qua21 = remove_prefix(qua21)

# STEP 3 — Check duplicates in each file 

all_dfs = {
    "dep11": dep11,
    "dep21": dep21,
    "hlt11": hlt11,
    "hlt21": hlt21,
    "eco11": eco11,
    "eco21": eco21,
    "qua11": qua11,
    "qua21": qua21,
}

any_found = False
for name, df in all_dfs.items():
    dupes = df[df["LA"].duplicated(keep=False)]["LA"].unique()
    if len(dupes) > 0:
        any_found = True
        print(f"\n  {name} has {len(dupes)} duplicate LA name(s):")
        for d in sorted(dupes):
            count = df[df["LA"] == d].shape[0]
            print(f"      '{d}'  appears {count} times")
    else:
        print(f" {name} — no duplicates")

if not any_found:
    print("\n  All files clean — no duplicates found")


# STEP 4 — Consistency check: which LA is missing from each file

all_las_union = set()
for df in all_dfs.values():
    all_las_union |= set(df["LA"].dropna())

non_la_patterns = r"(^in order|^east$|^west$|^north$|^south$|^nan$|^england|^wales|^total)"
all_las_union = {la for la in all_las_union
                 if not re.match(non_la_patterns, la.strip())}

print(f"\n  Total unique LA names across all files: {len(all_las_union)}")

for name, df in all_dfs.items():
    present   = set(df["LA"].dropna())
    missing   = sorted(all_las_union - present)
    extra     = sorted(present - all_las_union)
    print(f"  {name}  ({len(present)} LAs present)")
    if missing:
        print(f" MISSING ({len(missing)}):")
        for m in missing:
            print(f" - '{m}'")
    else:
        print(f" No missing LAs")

# 6. Define aggregate, spelling, boundary, and sum functions

In [ ]:
# CLEANING — Fix all 3 types of problems identified

regional_aggregates = {
    'east midlands', 'london', 'north east', 'north west',
    'south east', 'south west', 'west midlands',
    'yorkshire and the humber', 'east', 'wales', 'england',
    'england and wales'}

def drop_aggregates(df):
    """
        Remove rows where the 'LA' column contains a regional or national aggregate.

        These rows represent metadata or summary statistics (e.g., 'london', 'england',
        'wales') rather than individual local authorities. They are not needed for
        authority-level analysis and can cause spurious matches when merging datasets.

        Parameters
        ----------
        df : pd.DataFrame
            DataFrame that must contain a column named 'LA'.

        Returns
        -------
        pd.DataFrame
            A new DataFrame with all aggregate rows removed and the index reset.
            The original DataFrame is not modified.

        Notes
        -----
        The list of removed regions is hard-coded in the global set `regional_aggregates`.
        A message is printed listing any dropped values.
    """
    mask = df["LA"].isin(regional_aggregates)
    dropped = df[mask]["LA"].tolist()
    if dropped:
        print(f"  Dropping aggregates: {dropped}")
    return df[~mask].reset_index(drop=True)



spelling_fixes = {
    'rhondda cynon taff'    : 'rhondda cynon taf'}

def fix_spelling(df):
    df["LA"] = df["LA"].replace(spelling_fixes)
    return df

boundary_fixes_2011 = {
    'bournemouth' : 'bournemouth, christchurch and poole',
    'poole'       : 'bournemouth, christchurch and poole'}

las_to_drop_2011 = {
    'cumbria'}

def apply_boundary_fixes_2011(df):
    """
        Adjust 2011 local authority names to align with 2021 boundaries.

        Handles known boundary changes between 2011 and 2021:
        - Mergers: 'bournemouth' and 'poole' → 'bournemouth, christchurch and poole'
        - Splits: 'cumbria' are dropped because one 2011
        authority cannot be reliably split into multiple 2021 authorities.

        Parameters
        ----------
        df : pd.DataFrame
            A DataFrame (typically from a 2011 dataset) containing an 'LA' column.

        Returns
        -------
        pd.DataFrame
            DataFrame with renamed authorities (merged cases) and dropped rows for
            split authorities. Index is reset.

        Notes
        -----
        This function only renames authorities; it does NOT sum duplicate rows.
        Use `sum_duplicate_las()` afterwards to aggregate any rows that now share
        the same name (e.g., the former Bournemouth and Poole rows).
    """

    df["LA"] = df["LA"].replace(boundary_fixes_2011)

    dropped = df[df["LA"].isin(las_to_drop_2011)]["LA"].tolist()
    if dropped:
        print(f"  Dropping split LAs: {dropped}")
    df = df[~df["LA"].isin(las_to_drop_2011)]
    return df.reset_index(drop=True)

def sum_duplicate_las(df):
    """
        Aggregate numeric columns by local authority name, summing duplicate entries.

        After renaming merged authorities (e.g., 'bournemouth' and 'poole' both become
        'bournemouth, christchurch and poole'), this function combines their rows
        by summing all numeric columns.

        Parameters
        ----------
        df : pd.DataFrame
            DataFrame containing at least a 'LA' column and one or more numeric columns.
            Duplicate values in 'LA' are expected.

        Returns
        -------
        pd.DataFrame
            A new DataFrame with one row per unique local authority. All numeric
            columns are summed; non-numeric columns (other than 'LA') are discarded.
            Index is reset.

        Notes
        -----
        The grouping is performed only on numeric columns (detected via
        `df.select_dtypes(include="number")`). Non-numeric data (e.g., string descriptions)
        are not preserved, which is appropriate for the typical structure of these
        deprivation tables.
    """
    la_col   = df["LA"]
    num_cols = df.select_dtypes(include="number")
    merged   = num_cols.groupby(la_col).sum().reset_index()
    merged   = merged.rename(columns={"index": "LA"})
    return merged.reset_index(drop=True)


for name, df in [("dep11", dep11), ("hlt11", hlt11),
                 ("eco11", eco11), ("qua11", qua11)]:
    print(f"\n  {name}:")
    df = drop_aggregates(df)
    df = fix_spelling(df)
    df = apply_boundary_fixes_2011(df)
    df = sum_duplicate_las(df)
    print(f"  Final shape: {df.shape}")

    if   name == "dep11": dep11 = df
    elif name == "hlt11": hlt11 = df
    elif name == "eco11": eco11 = df
    elif name == "qua11": qua11 = df


for name, df in [("dep21", dep21), ("hlt21", hlt21),
                 ("eco21", eco21), ("qua21", qua21)]:
    print(f"\n  {name}:")
    df = drop_aggregates(df)    
    print(f"  Final shape: {df.shape}")
    if   name == "dep21": dep21 = df
    elif name == "hlt21": hlt21 = df
    elif name == "eco21": eco21 = df
    elif name == "qua21": qua21 = df

# 7. Define final sanity check function

In [ ]:
def check_cleaning(all_dfs):
    """
        Perform final sanity checks on cleaned deprivation DataFrames.

        This function runs a series of validation tests on a dictionary of DataFrames
        (typically the 2011 and 2021 datasets after cleaning). It prints results
        directly to the console and does not return any value. The checks include:

        1. Null and duplicate values in the 'LA' column.
        2. Building a reference set of unique LA names from all 2021 files.
        3. Consistency of LA sets across all 2021 files (they should be identical).
        4. Whether every LA in each 2011 file exists in the 2021 reference set.
        5. A quick numeric column type check (first numeric column should not be object).

        Parameters
        ----------
        all_dfs : dict of {str: pd.DataFrame}
            A dictionary where keys are dataset names (e.g., 'dep11', 'dep21') and
            values are pandas DataFrames. Each DataFrame must contain a column
            named 'LA' and at least one numeric column for the type check.

        Returns
        -------
        None
            The function prints results to stdout and returns nothing.

        Notes
        -----
        The reference set is built only from keys that contain the substring '21'
        (i.e., 2021 files). The function assumes that 2021 files represent the most
        up-to-date boundary set. For the numeric check, only the first numeric column
        is inspected as a sample; other columns are not verified.
    """
    for name, df in all_dfs.items():
        nulls = df["LA"].isna().sum()
        dupes = df["LA"].duplicated().sum()
        if nulls > 0:
            print(f" {name}: {nulls} null values in LA column")
        else:
            print(f" {name}: no nulls")
        if dupes > 0:
            print(f" {name}: {dupes} duplicate LA rows")
        else:
            print(f" {name}: no duplicates")

    ref_2021 = set()
    for name, df in all_dfs.items():
        if "21" in name:
            ref_2021.update(df["LA"].unique())
    print(f"\n 2. Reference set: {len(ref_2021)} unique LAs from 2021 files")

    print("\n 3. Consistency among 2021 files ")
    twentyone_sets = {}
    for name, df in all_dfs.items():
        if "21" in name:
            twentyone_sets[name] = set(df["LA"].unique())
    first_set = next(iter(twentyone_sets.values()))
    all_equal = True
    for name, s in twentyone_sets.items():
        if s != first_set:
            print(f" {name} set differs from reference")
            all_equal = False
    if all_equal:
        print(" All 2021 files contain exactly the same LA names")
    else:
        print(" 2021 files have mismatched LA sets - inner join may lose rows")

    print("\n 4. 2011 LAs ⊆ 2021 reference set")
    for name, df in all_dfs.items():
        if "11" in name:
            la_set = set(df["LA"].unique())
            missing = la_set - ref_2021
            if missing:
                print(f"  {name}: {len(missing)} LAs not in 2021 set:")
                for m in sorted(missing)[:5]:
                    print(f"  - '{m}'")
                if len(missing) > 5:
                    print(f" and {len(missing)-5} more")
            else:
                print(f" {name}: all {len(la_set)} LAs match 2021 names")

    for name, df in all_dfs.items():
        num_cols = df.select_dtypes(include="number").columns.tolist()
        if len(num_cols) == 0:
            print(f"{name}: no numeric columns found")
        else:
            col = num_cols[0]
            if df[col].dtype == "object":
                print(f" {name}: column '{col}' is still object dtype")
            else:
                print(f"{name}: numeric column '{col}' ok")

In [ ]:
# Re-build the dictionary after all cleaning steps
all_dfs_clean = {
    "dep11": dep11,
    "dep21": dep21,
    "hlt11": hlt11,
    "hlt21": hlt21,
    "eco11": eco11,
    "eco21": eco21,
    "qua11": qua11,
    "qua21": qua21}

check_cleaning(all_dfs_clean)

In [ ]:
def compare_column_names(pairs):
    """
        Compare column names between pairs of 2011 and 2021 DataFrames.

        For each pair (e.g., deprivation 2011 vs 2021), this function prints:
        - Total column counts for each DataFrame.
        - Number of common columns.
        - Columns present only in the 2011 version.
        - Columns present only in the 2021 version.
        - If column sets are identical, it also checks and reports whether the
        column order differs.

        Parameters
        ----------
        pairs : list of tuples
            Each tuple should contain four elements in the following order:
            (name_11, df_11, name_21, df_21)
            
            - name_11 : str
                Identifier for the 2011 DataFrame (e.g., 'dep11').
            - df_11 : pd.DataFrame
                The 2011 DataFrame to compare.
            - name_21 : str
                Identifier for the 2021 DataFrame (e.g., 'dep21').
            - df_21 : pd.DataFrame
                The 2021 DataFrame to compare.

        Returns
        -------
        None
            The function prints the comparison results directly to the console.

        Examples
        --------
        >>> pairs_to_check = [
        ...     ("dep11", dep11, "dep21", dep21),
        ...     ("hlt11", hlt11, "hlt21", hlt21),
        ... ]
        >>> compare_column_names(pairs_to_check)
    """
    for name11, df11, name21, df21 in pairs:
        cols11 = set(df11.columns)
        cols21 = set(df21.columns)
        
        only_in_11 = cols11 - cols21
        only_in_21 = cols21 - cols11
        common = cols11 & cols21
        
        print(f"\n{name11.upper()} vs {name21.upper()} ")
        print(f"  {name11}: {len(cols11)} columns")
        print(f"  {name21}: {len(cols21)} columns")
        print(f"  Common: {len(common)} columns")
        
        if only_in_11:
            print(f" Only in {name11} ({len(only_in_11)}):")
            for col in sorted(only_in_11)[:5]:
                print(f"      - '{col}'")
            if len(only_in_11) > 5:
                print(f" and {len(only_in_11)-5} more")
        else:
            print(f"No columns unique to {name11}")
            
        if only_in_21:
            print(f" Only in {name21} ({len(only_in_21)}):")
            for col in sorted(only_in_21)[:5]:
                print(f"      - '{col}'")
            if len(only_in_21) > 5:
                print(f" and {len(only_in_21)-5} more")
        else:
            print(f"No columns unique to {name21}")
            
        if not only_in_11 and not only_in_21:
            print(f"Column sets are identical")
            if list(df11.columns) != list(df21.columns):
                print(f" Column order differs between {name11} and {name21}")
            else:
                print(f" Column order is the same")

pairs_to_check = [
    ("dep11", dep11, "dep21", dep21),
    ("hlt11", hlt11, "hlt21", hlt21),
    ("eco11", eco11, "eco21", eco21),
    ("qua11", qua11, "qua21", qua21)]

compare_column_names(pairs_to_check)

In [ ]:
def standardise_dep_columns(df, year):
    """
        Rename columns in a deprivation (dep) DataFrame to a common set of names.

        This function handles variations in column names between the 2011 and 2021
        deprivation datasets (e.g., 'All categories: Classification of household deprivation'
        vs 'Total: All households' → both become 'Total'). After renaming, both
        DataFrames can be merged or compared more easily.

        Parameters
        ----------
        df : pd.DataFrame
            The deprivation DataFrame to rename.
        year : int
            The reference year (2011 or 2021). This parameter is currently not used
            but is kept for consistency with the other standardisation functions.

        Returns
        -------
        pd.DataFrame
            The renamed DataFrame (same object, modified in place and returned).

        Notes
        -----
        The mapping currently handles:
        - The total households column → 'Total'
        - Deprivation dimensions (1 to 4) with both numeric and word forms
        (e.g., 'deprived in 1 dimension' / 'one dimension' → 'Household is deprived in 1 dimension')
    """
    mapping = {
        'All categories: Classification of household deprivation': 'Total',
        'Total: All households': 'Total',
        'Household is deprived in 1 dimension': 'Household is deprived in 1 dimension',
        'Household is deprived in one dimension': 'Household is deprived in 1 dimension',
        'Household is deprived in 2 dimensions': 'Household is deprived in 2 dimensions',
        'Household is deprived in two dimensions': 'Household is deprived in 2 dimensions',
        'Household is deprived in 3 dimensions': 'Household is deprived in 3 dimensions',
        'Household is deprived in three dimensions': 'Household is deprived in 3 dimensions',
        'Household is deprived in 4 dimensions': 'Household is deprived in 4 dimensions',
        'Household is deprived in four dimensions': 'Household is deprived in 4 dimensions'}
    return df.rename(columns=mapping)

def standardise_hlt_columns(df, year):
    """
        Rename columns in a health (hlt) DataFrame to a common set of names.

        The 2011 and 2021 health tables use different names for the total resident
        column. This function standardises them to 'Total'.

        Parameters
        ----------
        df : pd.DataFrame
            The health DataFrame to rename.
        year : int
            The reference year (2011 or 2021). Not currently used in the mapping
            but kept for API consistency with other standardisation functions.

        Returns
        -------
        pd.DataFrame
            The renamed DataFrame (modified in place and returned).

        Examples
        --------
        >>> df = pd.DataFrame({'All categories: General health': [100]})
        >>> standardise_hlt_columns(df, 2011).columns[0]
        'Total'
    """
    mapping = {
        'All categories: General health': 'Total',
        'Total: All usual residents': 'Total'}
    return df.rename(columns=mapping)

def standardise_qua_columns(df, year):
    """
        Rename columns in a qualification (qua) DataFrame to a common set of names.

        This function handles variations in qualification column names between the
        2011 and 2021 datasets. The total column is standardised to 'Total', and
        specific levels are aligned (e.g., 'Level 1 qualifications' → 'Level 1 and entry level qualifications').

        Parameters
        ----------
        df : pd.DataFrame
            The qualification DataFrame to rename.
        year : int
            The reference year (2011 or 2021). Not used in the current mapping but
            retained for consistency.

        Returns
        -------
        pd.DataFrame
            The renamed DataFrame (modified in place and returned).

        Notes
        -----
        The mapping currently aligns:
        - Both 'All categories: Highest level of qualification' and
        'Total: All usual residents aged 16 years and over' → 'Total'
        - 'Level 1 qualifications' → 'Level 1 and entry level qualifications'
        - 'Level 4 qualifications and above' → 'Level 4 qualifications or above'
    """
    mapping = {
        'All categories: Highest level of qualification': 'Total',
        'Total: All usual residents aged 16 years and over': 'Total',
        'Level 1 qualifications': 'Level 1 and entry level qualifications',
        'Level 4 qualifications and above': 'Level 4 qualifications or above'}
    return df.rename(columns=mapping)

dep11 = standardise_dep_columns(dep11, 2011)
dep21 = standardise_dep_columns(dep21, 2021)

hlt11 = standardise_hlt_columns(hlt11, 2011)
hlt21 = standardise_hlt_columns(hlt21, 2021)

qua11 = standardise_qua_columns(qua11, 2011)
qua21 = standardise_qua_columns(qua21, 2021)


## Full Column Standardisation Table

### Deprivation (dep)

| Original 2011 Column Name | Original 2021 Column Name | Standardised Name (after renaming) |
|---------------------------|---------------------------|-------------------------------------|
| `LA` (identifier) | `LA` (identifier) | `LA` |
| `All categories: Classification of household deprivation` | `Total: All households` | `Total` |
| `Household is deprived in 1 dimension` | `Household is deprived in one dimension` | `Household is deprived in 1 dimension` |
| `Household is deprived in 2 dimensions` | `Household is deprived in two dimensions` | `Household is deprived in 2 dimensions` |
| `Household is deprived in 3 dimensions` | `Household is deprived in three dimensions` | `Household is deprived in 3 dimensions` |
| `Household is deprived in 4 dimensions` | `Household is deprived in four dimensions` | `Household is deprived in 4 dimensions` |

> **Note**: After renaming, both dataframes have the same 6 column names (LA + 5 numeric). No other columns exist.

---

### General Health (hlt)

| Original 2011 Column Name | Original 2021 Column Name | Standardised Name (after renaming) |
|---------------------------|---------------------------|-------------------------------------|
| `LA` (identifier) | `LA` (identifier) | `LA` |
| `All categories: General health` | `Total: All usual residents` | `Total` |
| `Very good health` | `Very good health` | `Very good health` |
| `Good health` | `Good health` | `Good health` |
| `Fair health` | `Fair health` | `Fair health` |
| `Bad health` | `Bad health` | `Bad health` |
| `Very bad health` | `Very bad health` | `Very bad health` |

> **Note**: Only the “total” column is renamed. All health categories are already identical.

---

### Highest Qualification (qua)

| Original 2011 Column Name | Original 2021 Column Name | Standardised Name (after renaming) |
|---------------------------|---------------------------|-------------------------------------|
| `LA` (identifier) | `LA` (identifier) | `LA` |
| `All categories: Highest level of qualification` | `Total: All usual residents aged 16 years and over` | `Total` |
| `No qualifications` | `No qualifications` | `No qualifications` |
| `Level 1 qualifications` | `Level 1 and entry level qualifications` | `Level 1 and entry level qualifications` |
| `Level 2 qualifications` | `Level 2 qualifications` | `Level 2 qualifications` |
| `Level 3 qualifications` | `Level 3 qualifications` | `Level 3 qualifications` |
| `Level 4 qualifications and above` | `Level 4 qualifications or above` | `Level 4 qualifications or above` |
| `Apprenticeship` | `Apprenticeship` | `Apprenticeship` |
| `Other qualifications` | `Other qualifications` | `Other qualifications` |

> **Note**:  
> - `Level 1 qualifications` (2011) is mapped to `Level 1 and entry level qualifications` (2021).  
> - `Level 4 qualifications and above` (2011) is mapped to `Level 4 qualifications or above` (2021).  
> - All other categories are unchanged.

---

## Summary of Changes

| Indicator | Number of columns (excluding LA) | Columns renamed | Columns identical |
|-----------|----------------------------------|----------------|-------------------|
| dep | 5 | 5 (all) | 0 |
| hlt | 6 | 1 | 5 |
| qua | 8 | 3 | 5 |

After applying the renaming code provided earlier, each pair (`dep11`/`dep21`, `hlt11`/`hlt21`, `qua11`/`qua21`) will have **identical column names** (though order may differ).

In [ ]:
# Re-run the comparison
pairs_to_check = [
    ("dep11", dep11, "dep21", dep21),
    ("hlt11", hlt11, "hlt21", hlt21),
    ("eco11", eco11, "eco21", eco21), 
    ("qua11", qua11, "qua21", qua21)]
compare_column_names(pairs_to_check)

In [ ]:
print("ECO11 columns:", list(eco11.columns))
print("ECO21 columns:", list(eco21.columns))
print("\nIn eco11 but not eco21:")
for col in set(eco11.columns) - set(eco21.columns):
    print(f"  - {col}")
print("\nIn eco21 but not eco11:")
for col in set(eco21.columns) - set(eco11.columns):
    print(f"  - {col}")

In [ ]:
def standardise_eco11(df):
    """
        Standardise the 2011 economic activity (eco) DataFrame to a common column structure.

        The 2011 dataset has columns in a specific order (based on the raw Excel layout).
        This function extracts the relevant columns by positional indices and creates
        a new DataFrame with renamed, standardised columns that match the structure
        expected for comparison with 2021 data.

        Parameters
        ----------
        df : pd.DataFrame
            The raw 2011 economic activity DataFrame, with the first column 'LA',
            column 1 as Total, and columns 2-16 containing specific economic categories
            in a predefined order.

        Returns
        -------
        pd.DataFrame
            A new DataFrame with the following columns:
            - 'LA' (local authority name)
            - 'Total' (all usual residents aged 16-74)
            - 'Economically active: Total' (sum of all active categories)
            - 'Economically active: Employed' (sum of employees and self-employed)
            - 'Economically active: Unemployed'
            - 'Economically active: Full-time student'
            - 'Economically inactive: Total'
            - 'Economically inactive: Retired'
            - 'Economically inactive: Student'
            - 'Economically inactive: Home or family'
            - 'Economically inactive: Long-term sick'
            - 'Economically inactive: Other'

        Notes
        -----
        The column indices used are based on the 2011 Excel layout:
        - col 2 : economically active total
        - col 3-8 : employed subcategories (part-time employees, full-time employees,
        self-employed with/without employees, etc.)
        - col 9 : unemployed
        - col 10 : full-time student
        - col 11 : economically inactive total
        - col 12 : retired
        - col 13 : student
        - col 14 : looking after home/family
        - col 15 : long-term sick/disabled
        - col 16 : other inactive
    """
    out = pd.DataFrame()
    out["LA"]                                    = df["LA"]
    out["Total"]                                 = df.iloc[:, 1]
    out["Economically active: Total"]            = df.iloc[:, 2]
    out["Economically active: Employed"]         = (df.iloc[:, 3] + df.iloc[:, 4] +df.iloc[:, 5] + df.iloc[:, 6] +
                                                    df.iloc[:, 7] + df.iloc[:, 8])
    out["Economically active: Unemployed"]       = df.iloc[:, 9]
    out["Economically active: Full-time student"]= df.iloc[:, 10]
    out["Economically inactive: Total"]          = df.iloc[:, 11]
    out["Economically inactive: Retired"]        = df.iloc[:, 12]
    out["Economically inactive: Student"]        = df.iloc[:, 13]
    out["Economically inactive: Home or family"] = df.iloc[:, 14]
    out["Economically inactive: Long-term sick"] = df.iloc[:, 15]
    out["Economically inactive: Other"]          = df.iloc[:, 16]
    return out

def standardise_eco21(df):
    """
        Standardise the 2021 economic activity (eco) DataFrame to a common column structure.

        The 2021 dataset splits economic activity into two groups:
        - Economically active (excluding full-time students)
        - Economically active full-time students
        This function aggregates appropriate subcategories to create columns
        that are directly comparable with the standardised 2011 DataFrame.

        Parameters
        ----------
        df : pd.DataFrame
            The raw 2021 economic activity DataFrame, with the first column 'LA',
            column 1 as Total, and columns 2-13 containing specific economic categories
            in a predefined order.

        Returns
        -------
        pd.DataFrame
            A new DataFrame with the same column names as produced by `standardise_eco11`:
            - 'LA'
            - 'Total'
            - 'Economically active: Total' (sum of active excl. students + active students)
            - 'Economically active: Employed' (sum of employed from both groups)
            - 'Economically active: Unemployed' (sum of unemployed from both groups)
            - 'Economically active: Full-time student' (active full-time students)
            - 'Economically inactive: Total'
            - 'Economically inactive: Retired'
            - 'Economically inactive: Student'
            - 'Economically inactive: Home or family'
            - 'Economically inactive: Long-term sick'
            - 'Economically inactive: Other'

        Notes
        -----
        The column indices used are based on the 2021 Excel layout:
        - col 2 : economically active (excluding full-time students) total
        - col 3 : employed (in that group)
        - col 4 : unemployed (in that group)
        - col 5 : economically active full-time students total
        - col 6 : employed (in that group)
        - col 7 : unemployed (in that group)
        - col 8 : economically inactive total
        - col 9 : retired
        - col 10 : student
        - col 11 : looking after home/family
        - col 12 : long-term sick/disabled
        - col 13 : other inactive
    """
    out = pd.DataFrame()
    out["LA"]                                    = df["LA"]
    out["Total"]                                 = df.iloc[:, 1]
    out["Economically active: Total"]            = df.iloc[:, 2] + df.iloc[:, 5]
    out["Economically active: Employed"]         = df.iloc[:, 3] + df.iloc[:, 6]
    out["Economically active: Unemployed"]       = df.iloc[:, 4] + df.iloc[:, 7]
    out["Economically active: Full-time student"]= df.iloc[:, 5]
    out["Economically inactive: Total"]          = df.iloc[:, 8]
    out["Economically inactive: Retired"]        = df.iloc[:, 9]
    out["Economically inactive: Student"]        = df.iloc[:, 10]
    out["Economically inactive: Home or family"] = df.iloc[:, 11]
    out["Economically inactive: Long-term sick"] = df.iloc[:, 12]
    out["Economically inactive: Other"]          = df.iloc[:, 13]
    return out

eco11 = standardise_eco11(eco11)
eco21 = standardise_eco21(eco21)

print("eco11 columns:", eco11.columns.tolist())
print("eco21 columns:", eco21.columns.tolist())
print("\nColumn match:", eco11.columns.tolist() == eco21.columns.tolist())
print("\neco11 sample:")
print(eco11.head(2))
print("\neco21 sample:")
print(eco21.head(2))

In [ ]:
for name, df in [("eco11", eco11), ("eco21", eco21)]:
    df["_check"] = df["Economically active: Total"] + df["Economically inactive: Total"]
    df["_diff"]  = (df["_check"] - df["Total"]).abs()
    max_diff     = df["_diff"].max()
    bad_rows     = df[df["_diff"] > 10]
    print(f"{name}: max difference = {max_diff:.0f}")
    if not bad_rows.empty:
        print(f" Rows with large discrepancy:")
        print(bad_rows[["LA", "Total", "Economically active: Total",
                         "Economically inactive: Total", "_diff"]])
    else:
        print(f" All rows balance correctly")
    df.drop(columns=["_check", "_diff"], inplace=True)

In [ ]:
eco11

In [ ]:
eco21

In [ ]:
def add_poor_health(df):
    """
        Combine 'bad health' and 'very bad health' percentage columns into a single 'poor_health' column.

        This function searches for column names containing 'bad health' (but not 'very')
        and 'very bad health', sums them, and stores the result in a new column named
        'poor_health'. The result is rounded to two decimal places.

        Parameters
        ----------
        df : pd.DataFrame
            DataFrame expected to contain health-related columns, typically from the
            health (hlt) dataset after percentage conversion. Column names are matched
            case-insensitively.

        Returns
        -------
        pd.DataFrame
            The same DataFrame with an additional 'poor_health' column (modified in place).
            If the required columns are not found, a message is printed and the DataFrame
            is returned unchanged.

        Notes
        -----
        This function assumes that the input DataFrame has already been converted to
        percentages (i.e., values represent percentages of the total population).
        Column detection is based on substrings; the first match for each category is used.
    """
    
    bad_col     = [c for c in df.columns if "bad health" in c.lower() 
                   and "very" not in c.lower()]
    very_bad_col = [c for c in df.columns if "very bad health" in c.lower()]

    if bad_col and very_bad_col:
        df["poor_health"] = df[bad_col[0]] + df[very_bad_col[0]]
        df["poor_health"] = df["poor_health"].round(2)
        print(f"poor_health_pct created")
    else:
        print(f"Could not find bad/very bad columns")
        print(f"Available: {df.columns.tolist()}")
    return df

hlt11 = add_poor_health(hlt11)
hlt21 = add_poor_health(hlt21)

print("\nhlt11_pct sample:")
print(hlt11[["LA", "poor_health"]].head(3))

In [ ]:
def convert_to_pct(df, total_col="Total"):
    """
        Convert count columns to percentages of a total column.

        Every numeric column other than the LA identifier and the specified total column
        is divided by the total column and multiplied by 100. The original total column
        is retained as is. New columns are named with a '_pct' suffix.

        Parameters
        ----------
        df : pd.DataFrame
            DataFrame containing at least an 'LA' column, a total column (default 'Total'),
            and one or more numeric count columns to convert.
        total_col : str, default 'Total'
            Name of the column that holds the denominator (total population / households).

        Returns
        -------
        pd.DataFrame
            A new DataFrame with the following columns:
            - 'LA' (unchanged)
            - total_col (unchanged)
            - For each numeric count column 'col', a new column 'col_pct' = (col / total) * 100,
            rounded to two decimal places.

        Examples
        --------
        >>> df = pd.DataFrame({'LA': ['A'], 'Total': [100], 'Deprived': [25]})
        >>> convert_to_pct(df)
        LA  Total  Deprived_pct
        0   A    100         25.00
    """
    out = pd.DataFrame()
    out["LA"]    = df["LA"]
    out["Total"] = df[total_col]

    numeric_cols = [c for c in df.columns if c not in ["LA", total_col]]

    for col in numeric_cols:
        pct_name      = col + "_pct"
        out[pct_name] = (df[col] / df[total_col] * 100).round(2)

    return out


# Apply to all dataframes
dep11 = convert_to_pct(dep11)
dep21 = convert_to_pct(dep21)
hlt11 = convert_to_pct(hlt11)
hlt21 = convert_to_pct(hlt21)
eco11 = convert_to_pct(eco11)
eco21 = convert_to_pct(eco21)
qua11 = convert_to_pct(qua11)
qua21 = convert_to_pct(qua21)


In [ ]:
hlt11

In [ ]:

def calc_deprivation_score(df):
    """
        Calculate a weighted deprivation score from the four deprivation dimension percentages.

        The score is computed as a weighted average of the percentages of households
        deprived in 1, 2, 3, and 4 dimensions:
            score = (pct1*1 + pct2*2 + pct3*3 + pct4*4) / 4
        A higher score indicates a more deprived local authority (range roughly 0-100).

        Parameters
        ----------
        df : pd.DataFrame
            DataFrame that must contain the following percentage columns:
            - 'Household is deprived in 1 dimension_pct'
            - 'Household is deprived in 2 dimensions_pct'
            - 'Household is deprived in 3 dimensions_pct'
            - 'Household is deprived in 4 dimensions_pct'

        Returns
        -------
        pd.DataFrame
            A copy of the input DataFrame with an additional column:
            'deprivation_score' (float, rounded to two decimal places).
            If any required column is missing, the original DataFrame is returned
            unchanged and an error message is printed.

        Notes
        -----
        The calculation assumes that the percentages sum to 100 when including the
        'not deprived' category, but that category is not used in the weighted score.
    """
    required = [
        "Household is deprived in 1 dimension_pct",
        "Household is deprived in 2 dimensions_pct",
        "Household is deprived in 3 dimensions_pct",
        "Household is deprived in 4 dimensions_pct"]

    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"  Missing columns: {missing}")
        print(f"  Available columns: {df.columns.tolist()}")
        return df

    df = df.copy()
    df["deprivation_score"] = (
        df["Household is deprived in 1 dimension_pct"] * 1 +
        df["Household is deprived in 2 dimensions_pct"] * 2 +
        df["Household is deprived in 3 dimensions_pct"] * 3 +
        df["Household is deprived in 4 dimensions_pct"] * 4
    ).round(2) / 4

    return df


dep11 = calc_deprivation_score(dep11)
dep21 = calc_deprivation_score(dep21)

print("✓ Deprivation score calculated")
print("\ndep11_pct — top 5 most deprived LAs:")
print(dep11[["LA", "deprivation_score"]].sort_values(
      "deprivation_score", ascending=False).head(5).to_string(index=False))

print("\ndep21_pct — top 5 most deprived LAs:")
print(dep21[["LA", "deprivation_score"]].sort_values(
      "deprivation_score", ascending=False).head(5).to_string(index=False))

In [ ]:
def check_pct_sum(df, name, pct_cols):
    """
        Verify that a set of percentage columns sum to approximately 100 for every row.

        This check is useful for ensuring that after converting counts to percentages,
        the component parts (e.g., deprivation dimensions or health categories) still
        add up to the total (≈100%). Rows where the sum deviates more than 1% from 100
        (i.e., outside the range 99-101) are flagged.

        Parameters
        ----------
        df : pd.DataFrame
            DataFrame containing the percentage columns to check.
        name : str
            Descriptive name for the dataset (used only in print statements).
        pct_cols : list of str
            List of column names that should sum to 100% per row.

        Returns
        -------
        None
            The function prints the result of the check to the console. It does not
            modify the DataFrame (a temporary '_sum' column is added and then removed).

        Notes
        -----
        The tolerance of ±1% accounts for small rounding errors. If the sum is outside
        [99, 101], the function prints the affected rows. Otherwise, it prints a
        success message.
    """
    df["_sum"] = df[pct_cols].sum(axis=1)
    bad = df[df["_sum"].between(99, 101) == False]
    if bad.empty:
        print(f"{name} — all rows sum to ~100%")
    else:
        print(f"{name} — {len(bad)} rows don't sum to 100:")
        print(bad[["LA", "_sum"]])
    df.drop(columns=["_sum"], inplace=True)
dep_pct_cols = [
    "Household is not deprived in any dimension_pct",
    "Household is deprived in 1 dimension_pct",
    "Household is deprived in 2 dimensions_pct",
    "Household is deprived in 3 dimensions_pct",
    "Household is deprived in 4 dimensions_pct"]

hlt_pct_cols = [c for c in hlt11.columns if c.endswith("_pct")]

print("Deprivation pct sums:")
check_pct_sum(dep11, "dep11_pct", dep_pct_cols)
check_pct_sum(dep21, "dep21_pct", dep_pct_cols)

print("\nHealth pct sums:")
check_pct_sum(hlt11, "hlt11_pct", hlt_pct_cols)
check_pct_sum(hlt21, "hlt21_pct", hlt_pct_cols)